In [1]:
from xaikd import models

In [2]:
models.get_trained_model("imagenet-vgg16-tv")

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [3]:
models.get_trained_model("cifar100-vgg11-v1")

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): ReLU(inplace=True)
    (13): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (14): ReLU(inplace=True)
    (15): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
 

In [4]:
models.get_untrained_model("vggcustomimagenetdims-32-24-24-10", num_classes=10)

Sequential(
  (stem): Sequential(
    (0): ConvBN(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ReLU()
    (2): ConvBN(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): ConvBN(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): ReLU()
    (2): ConvBN(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_runni

In [5]:
from collections import OrderedDict
from copy import deepcopy
from torch import nn
import numpy as np 
import torch
from xaikd.utils.modules import merge_convKxK_and_conv1x1


def merge_bn_linear(bn, linear):
    
    bn_mean = bn.running_mean.clone()
    bn_std = (bn.running_var.clone() + bn.eps) ** 0.5

    if bn.affine:
        bn_scale = bn.weight.clone()
        bn_shift = bn.bias.clone()
    else:
        bn_scale = 1
        bn_shift = 0

    W_bn = torch.diag(bn_scale / bn_std)

    b_bn = -(bn_scale / bn_std) * bn_mean + bn_shift
    
    W = linear.weight
    b = linear.bias
    
    new_linear = deepcopy(linear)
    
    new_linear.weight = W @ W_bn
    new_linear.bias = W @ b_bn  + b
    
    return new_linear
    
    

def merging_model(model):
    
    features = []
          
    for layer_name in ["stem", "layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name)
        for modul in layer.children():
            if hasattr(modul, "canonize"):
                features.append(modul.canonize())
            else:
                features.append(modul)
                
    merged_features = []
    
    arr_cls_modules = list(model.classifier.children())

    last_adapter = features[-1]

    
    features = features[:-1]
        
    for fix in range(len(features)):
        modul = features[fix]
        if fix < len(features)-2:
            next_modul = features[fix+1]
        else:
            next_modul = None
        
        if (isinstance(modul, nn.Conv2d) and modul.kernel_size[0] > 1) \
            and (isinstance(next_modul, nn.Conv2d) and next_modul.kernel_size[0] == 1):
            
            merged_features.append(merge_convKxK_and_conv1x1(modul, next_modul))
        elif isinstance(modul, nn.Conv2d) and modul.kernel_size[0] == 1:
            continue
        else:
            merged_features.append(modul)
            
            
    
            
    arr_cls_modules = list(model.classifier.children())
    

    merged_conv = merge_convKxK_and_conv1x1(last_adapter, arr_cls_modules[0])
    
                            
    return nn.Sequential(
        OrderedDict(
            [
                ("features", nn.Sequential(*merged_features)),
                ("classifier", nn.Sequential(merged_conv, *arr_cls_modules[1:]))
            ]
        )
    )

@torch.no_grad()
def ano():
    torch.manual_seed(1)
    x_train = torch.rand(5, 3, 224, 224)

    model = models.get_untrained_model("vggcustomimagenetdims-32-24-24-10", num_classes=10)
    model(x_train)
    
    model.eval()
    
    canonized_model = merging_model(model)
    
    canonized_model.eval()
    
    x = torch.rand(5, 3, 224, 224)

    expected = model(x)
    actual = canonized_model(x)
    
    np.testing.assert_allclose(actual, expected, atol=1e-6)
    print(f"all testt passed! ")
    
    return canonized_model

ano()

all testt passed! 


Sequential(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU()
    (7): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): Conv2d(32, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU()
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (17): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding

In [6]:
[1, 2, 3][:-1]

[1, 2]